# Portfolio Optimization: The Efficient Frontier

## Summary

| | |
| :--- | :--- |
| **Idea** | Weight the holdings by mean-variance optimisation (maximum Sharpe ratio, or minimum volatility) to beat an equal-weight portfolio. |
| **Verdict** | Rejected for now |
| **Result** | In sample, the maximum-Sharpe portfolio showed a 48% return at 35% volatility, with 52% of the money in NVDA. Out of sample, weights fitted on history up to a year ago returned 1.03x over the following 12 months against 1.07x for equal weight. |
| **Trust** | Low to medium. One 12-month out-of-sample window on 14 stocks. The pattern is the well-known one, though: the optimiser concentrates in whatever rose most in the fitting period and does not beat equal weight afterwards. |
| **Date** | 2026-07-19 |
| **Code status** | Frozen: written before the current modules and no longer runs. Read it as a report. |

*Summary added 2026-10-13 by Claude from this notebook's own commentary and stored results. The verdict is a draft for Ben to confirm or correct.*

# Portfolio Optimization: The Efficient Frontier
This notebook explores how to allocate capital across a set of assets to either minimize risk (volatility) or maximize return. 

### Key Concepts:
1. **Expected Return**: The weighted average of the individual assets' historical returns.
2. **Portfolio Volatility**: The standard deviation of the portfolio, which accounts for the correlations between assets.
3. **The Efficient Frontier**: A curve representing the set of optimal portfolios that offer the highest expected return for a defined level of risk.
4. **Sharpe Ratio**: A measure of risk-adjusted return ($Return / Volatility$).

In [15]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize

# Import our foundation classes
from PriceData import PriceData
from PriceIndicators import PriceIndicators

# Set up data access
pd.options.display.max_columns = None
price_loader = PriceData("YF/yf-prices.csv")
indicators = PriceIndicators()

# Load tickers from our watchlist (De Giro or HL)
watchlist = pd.read_csv('degiro.csv')
tickers = watchlist['ticker'].tolist()

print(f"Loaded {len(tickers)} tickers from watchlist.")

Loaded 14 tickers from watchlist.


## 1. Prepare Data and Calculate Statistics
We'll fetch the historical wide-format price data and calculate daily returns. From these, we derive the annualized mean return and the covariance matrix.

In [16]:
# Fetch wide data (dates as index, tickers as columns)
prices = price_loader.get_data_wide(tickers=tickers)

# Calculate Daily Returns
returns = indicators.daily_returns(prices).dropna()

# Annualized Statistics (assuming 252 trading days)
annual_returns = returns.mean() * 252
cov_matrix = returns.cov() * 252

print("Annualized Returns (First 5):")
print(annual_returns.head())

Annualized Returns (First 5):
ticker
AAPL     0.320540
AMZN     0.192870
BABA     0.058534
BLDR     0.340711
GOOGL    0.300084
dtype: float64


## 2. Defining Portfolio Performance
We need a way to calculate the expected return and volatility for any given set of weights.

In [17]:
def portfolio_performance(weights, returns, cov_matrix):
    portfolio_return = np.sum(returns * weights)
    portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return portfolio_return, portfolio_volatility

def neg_sharpe_ratio(weights, returns, cov_matrix, risk_free_rate=0.02):
    p_ret, p_vol = portfolio_performance(weights, returns, cov_matrix)
    return -(p_ret - risk_free_rate) / p_vol

def portfolio_volatility(weights, returns, cov_matrix):
    return portfolio_performance(weights, returns, cov_matrix)[1]

## 3. Finding the "Best" Portfolios
We'll find two specific points on the frontier:
1. **Maximum Sharpe Ratio**: The portfolio with the best risk-adjusted return.
2. **Minimum Volatility**: The safest possible combination of these assets.

In [18]:
num_assets = len(tickers)
args = (annual_returns, cov_matrix)
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
bounds = tuple((0.0, 1.0) for asset in range(num_assets))

# 1. Maximize Sharpe Ratio
max_sharpe_res = minimize(neg_sharpe_ratio, num_assets * [1./num_assets], args=args,
                        method='SLSQP', bounds=bounds, constraints=constraints)

# 2. Minimize Volatility
min_vol_res = minimize(portfolio_volatility, num_assets * [1./num_assets], args=args,
                      method='SLSQP', bounds=bounds, constraints=constraints)

# Extract results
weights_ms = max_sharpe_res.x
ret_ms, vol_ms = portfolio_performance(weights_ms, annual_returns, cov_matrix)

weights_mv = min_vol_res.x
ret_mv, vol_mv = portfolio_performance(weights_mv, annual_returns, cov_matrix)

print(f"Max Sharpe: Return {ret_ms:.2%}, Volatility {vol_ms:.2%}")
print(f"Min Volatility: Return {ret_mv:.2%}, Volatility {vol_mv:.2%}")

Max Sharpe: Return 48.03%, Volatility 34.67%
Min Volatility: Return 18.74%, Volatility 21.76%


## 4. Calculating the Efficient Frontier
To draw the line, we'll find the minimum volatility for a sequence of target returns.

In [19]:
target_returns = np.linspace(ret_mv, annual_returns.max(), 30)
efficient_vols = []

for target in target_returns:
    cons = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
            {'type': 'eq', 'fun': lambda x: portfolio_performance(x, annual_returns, cov_matrix)[0] - target})
    
    res = minimize(portfolio_volatility, num_assets * [1./num_assets], args=args,
                  method='SLSQP', bounds=bounds, constraints=cons)
    efficient_vols.append(res.fun)

print("Calculated Efficient Frontier points.")


Calculated Efficient Frontier points.


## 5. Visualizing the Frontier
Now we visualize the risk-return profiles of individual tickers, the calculated efficient frontier, and our two optimal portfolios.

In [20]:
fig = go.Figure()

# Individual Tickers
fig.add_trace(go.Scatter(
    x=np.sqrt(np.diag(cov_matrix)), 
    y=annual_returns,
    mode='markers+text',
    name='Individual Assets',
    text=tickers,
    textposition="top center",
    marker=dict(size=10, color='gray', opacity=0.5)
))

# Efficient Frontier Line
fig.add_trace(go.Scatter(
    x=efficient_vols, 
    y=target_returns,
    mode='lines',
    name='Efficient Frontier',
    line=dict(color='black', dash='dash')
))

# Max Sharpe Portfolio
fig.add_trace(go.Scatter(
    x=[vol_ms], y=[ret_ms],
    mode='markers',
    name='Max Sharpe Ratio',
    marker=dict(color='red', size=15, symbol='star')
))

# Min Volatility Portfolio
fig.add_trace(go.Scatter(
    x=[vol_mv], y=[ret_mv],
    mode='markers',
    name='Minimum Volatility',
    marker=dict(color='green', size=15, symbol='diamond')
))

fig.update_layout(
    title='Efficient Frontier: Risk vs. Expected Return',
    xaxis_title='Annualized Volatility (Standard Deviation)',
    yaxis_title='Annualized Expected Return',
    template='plotly_white',
    height=600
)

fig.show()

## 6. Suggested Allocations
Below are the suggested weights for each ticker under the two different strategies. Notice how the Minimum Volatility strategy often favors less correlated or lower-risk assets, whereas the Max Sharpe strategy looks for the best balance.


In [21]:
# Create a summary DataFrame of weights
allocation_df = pd.DataFrame({
    'Ticker': tickers,
    'Max Sharpe Weights (%)': (weights_ms * 100).round(2),
    'Min Volatility Weights (%)': (weights_mv * 100).round(2)
})

# Filter out zero-weight tickers to keep it clean
active_allocations = allocation_df[(allocation_df['Max Sharpe Weights (%)'] > 0.01) | 
                                   (allocation_df['Min Volatility Weights (%)'] > 0.01)]

print("Top Allocations for both strategies:")
print(active_allocations.sort_values(by='Max Sharpe Weights (%)', ascending=False))


Top Allocations for both strategies:
   Ticker  Max Sharpe Weights (%)  Min Volatility Weights (%)
10   NVDA                   51.87                        0.00
0    BABA                   21.88                        2.24
7    MSFT                   16.39                       41.10
4    BLDR                    7.08                        7.75
3    AAPL                    2.78                        0.00
2    AMZN                    0.00                        5.95
1   GOOGL                    0.00                        4.83
5      MA                    0.00                       14.69
8    MNST                    0.00                       10.72
9    NFLX                    0.00                        5.69
13   ULTA                    0.00                        7.03


## 7. Backtesting the Optimization Logic
To see if these insights are actually useful, we'll perform an "Out-of-Sample" test. 
1. **Training Period**: We'll calculate the optimal weights using data up until 1 year ago.
2. **Testing Period**: We'll then see how those specific weights would have performed over the *last* 12 months compared to an equal-weighted portfolio.


In [22]:
# 1. Split data into "In-Sample" (Training) and "Out-of-Sample" (Testing)
# We'll use everything up to 1 year ago to find our weights
test_cutoff = prices.index.max() - pd.DateOffset(years=1)
train_prices = prices[prices.index <= test_cutoff]
test_prices = prices[prices.index > test_cutoff]

# 2. Calculate Statistics for the Training Period
train_returns = train_prices.pct_change().dropna()
train_ann_ret = train_returns.mean() * 252
train_cov = train_returns.cov() * 252

# 3. Find Weights (In-Sample)
args_train = (train_ann_ret, train_cov)
res_ms_train = minimize(neg_sharpe_ratio, num_assets * [1./num_assets], args=args_train,
                        method='SLSQP', bounds=bounds, constraints=constraints)

weights_bt = res_ms_train.x

# 4. Compare Performance in the Testing Period (Last 12 Months)
test_returns = test_prices.pct_change().dropna()
portfolio_test_returns = (test_returns * weights_bt).sum(axis=1)
equal_weight_returns = test_returns.mean(axis=1)

# Cumulative Returns
cum_opt = (1 + portfolio_test_returns).cumprod()
cum_eq = (1 + equal_weight_returns).cumprod()

# 5. Visualize
fig_bt = go.Figure()
fig_bt.add_trace(go.Scatter(x=cum_opt.index, y=cum_opt, name='Optimized Portfolio (Max Sharpe)'))
fig_bt.add_trace(go.Scatter(x=cum_eq.index, y=cum_eq, name='Equal Weighted Portfolio', line=dict(dash='dash')))

fig_bt.update_layout(title='Backtest: Last 12 Months Performance (Weights based on prior data)',
                   yaxis_title='Cumulative Return', template='plotly_white')
fig_bt.show()

print(f"Final Return - Optimized: {cum_opt.iloc[-1]:.2f}x")
print(f"Final Return - Equal Weight: {cum_eq.iloc[-1]:.2f}x")


Final Return - Optimized: 1.03x
Final Return - Equal Weight: 1.07x
